# Tutorial 2b — Cell–cell communication in Xenium breast cancer

**Spatial omics hands-on — Google Colab tutorial**

A ligand in one cell and a receptor in another cell are a **candidate conversation**, not
proof that signalling occurred. A candidate is more plausible when:

1. the ligand is expressed in a plausible **sender** population;
2. the receptor is expressed in a plausible **receiver** population; and
3. those populations are sufficiently close in the tissue.

This tutorial uses [**LIANA+**](https://www.nature.com/articles/s41556-024-01469-w) and a
public human breast-cancer **10x Xenium** dataset from
[Janesick *et al.*](https://www.nature.com/articles/s41467-023-43458-x). The compressed
download is approximately **65 MB**. It contains 118,752 cells and 313 targeted genes; we
analyse a fixed spatial crop of about 9,800 cells so the workshop runs quickly on free Colab.

**Learning outcomes**

- distinguish expression evidence, cell-type proximity and local spatial coordination;
- run spatially weighted LIANA+ consensus inference;
- map where a candidate interaction occurs, including **CXCL12–CXCR4**; and
- recognise the limitations of targeted panels and marker-derived teaching labels.

> **Important.** These analyses are hypothesis-generating. RNA abundance is not protein
> abundance, receptor activation or causal evidence.

## 0. Setup

Run the installation cell once in each fresh Colab runtime. Installation normally takes a
few minutes. `pandas<3` follows current LIANA+ compatibility requirements.

In [ ]:
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = [
    "liana>=1.9,<2",
    "squidpy>=1.8,<2",
    "scanpy>=1.12,<2",
    "seaborn>=0.13,<1",
    "pandas<3",
]

if IN_COLAB:
    print("Installing LIANA+ and spatial-analysis dependencies ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PACKAGES],
                   check=True)
    print("Done")
else:
    print("Not running in Colab — assuming the packages are installed.")

In [ ]:
import os
import time
import urllib.request
import warnings
from importlib.metadata import version
from pathlib import Path

import liana as li
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from IPython.display import display
from scipy import sparse
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 0
np.random.seed(SEED)
sns.set_theme(style="white", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200})

print("liana  :", version("liana"))
print("scanpy :", version("scanpy"))
print("pandas :", pd.__version__)

## 1. Download and inspect the breast-cancer Xenium data

We use replicate 2 from the public
[Hugging Face mirror](https://huggingface.co/datasets/Shaow/humanbreast_xenium_janesick).
The `.h5ad` file contains a sparse count matrix and spatial coordinates. It has no curated
cell-type labels, so Section 2 creates deliberately simple **teaching annotations**.

The first run downloads about 65 MB. Colab keeps the file only for the current runtime.

In [ ]:
DATA_URL = (
    "https://huggingface.co/datasets/Shaow/humanbreast_xenium_janesick/"
    "resolve/main/h5ad/adata_xenium_rep2.h5ad"
)
override = os.environ.get("LIANA_BREAST_DATA")  # useful for local/offline validation
data_path = Path(override) if override else Path("data/adata_xenium_rep2.h5ad")
data_path.parent.mkdir(parents=True, exist_ok=True)

if not data_path.exists():
    print("Downloading approximately 65 MB ...")
    urllib.request.urlretrieve(DATA_URL, data_path)
else:
    print("Using cached data:", data_path)

t0 = time.time()
adata_all = sc.read_h5ad(data_path)
print(f"Loaded in {time.time() - t0:.1f} s")
print(adata_all)
print("matrix type:", type(adata_all.X).__name__)
print("spatial coordinate range:",
      adata_all.obsm["spatial"].min(axis=0).round(1), "to",
      adata_all.obsm["spatial"].max(axis=0).round(1))

### 1.1 Use a reproducible, information-rich crop

The crop below was selected before inference because it contains tumour, immune, vascular
and stromal regions. Cropping reduces computation; it does not reduce the 65 MB download.
The supplied Xenium coordinates are treated as micrometres for the distance calculations.

In [ ]:
SPATIAL_KEY = "spatial"
X0, Y0, CROP_SIZE = -6500, -3400, 1800
xy_all = adata_all.obsm[SPATIAL_KEY]
in_crop = (
    (xy_all[:, 0] >= X0) & (xy_all[:, 0] < X0 + CROP_SIZE) &
    (xy_all[:, 1] >= Y0) & (xy_all[:, 1] < Y0 + CROP_SIZE)
)

fig, ax = plt.subplots(figsize=(8.5, 5.6))
ax.scatter(xy_all[:, 0], xy_all[:, 1], s=0.25, color="#C7C7C7", rasterized=True)
ax.add_patch(plt.Rectangle((X0, Y0), CROP_SIZE, CROP_SIZE, fill=False,
                           edgecolor="#C1272D", linewidth=2.2))
ax.set(aspect="equal", title=f"Full replicate and analysis crop ({in_crop.sum():,} cells)",
       xlabel="spatial x (µm)", ylabel="spatial y (µm)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

adata = adata_all[in_crop].copy()
del adata_all
print(f"working crop: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

## 2. Normalise and create transparent teaching annotations

The dataset does not provide curated labels. We score short marker sets, assign the
highest-scoring class and flag weak or nearly tied assignments as `Other / ambiguous`.
This is convenient for teaching but **not a validated breast-cancer annotation workflow**.
For research, use pathology, a validated reference-mapping method, multiple markers and
manual review. In particular, `Tumour luminal` and `Tumour basal` mean epithelial-expression
programmes here; copy-number or pathology evidence would be required to call malignant cells.

In [ ]:
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

MARKERS = {
    "Tumour luminal": ["EPCAM", "KRT8", "KRT7", "ERBB2", "ESR1", "PGR", "GATA3",
                       "FOXA1", "SCGB2A1", "CEACAM6", "AGR3"],
    "Tumour basal": ["KRT5", "KRT14", "KRT15", "KRT16", "KRT6B", "CLCA2"],
    "T cell": ["CD3D", "CD3E", "CD3G", "TRAC", "CD247", "CD8A", "CD8B", "IL7R"],
    "B / plasma": ["CD19", "CD79A", "CD79B", "MS4A1", "CD27", "MZB1", "DERL3", "TNFRSF17"],
    "Myeloid": ["LYZ", "FCER1G", "TYROBP", "CD68", "CD14", "AIF1", "C1QA", "C1QC", "APOC1"],
    "Fibroblast": ["LUM", "PDGFRA", "DPT", "CCDC80", "PCOLCE", "SFRP1", "SFRP4", "POSTN", "LRRC15"],
    "Endothelial": ["PECAM1", "VWF", "KDR", "CLDN5", "EGFL7", "SOX17", "SOX18", "CLEC14A", "CD93"],
    "Perivascular": ["PDGFRB", "ACTA2", "MYH11", "MYLK", "NDUFA4L2", "AVPR1A"],
    "Mast": ["KIT", "CPA3", "TPSAB1", "HDC"],
    "NK": ["NKG7", "GNLY", "KLRD1", "KLRC1", "KLRF1", "PRF1"],
    "Adipocyte": ["ADIPOQ", "LEP", "PPARG", "UCP1", "ADH1B"],
}

scores = {}
for label, marker_list in MARKERS.items():
    present = [g for g in marker_list if g in adata.var_names]
    scores[label] = np.asarray(adata[:, present].X.mean(axis=1)).ravel()

score_table = pd.DataFrame(scores, index=adata.obs_names)
ordered = np.sort(score_table.to_numpy(), axis=1)
labels = score_table.idxmax(axis=1).astype(object)
ambiguous = (ordered[:, -1] < 0.25) | ((ordered[:, -1] - ordered[:, -2]) < 0.03)
labels.loc[ambiguous] = "Other / ambiguous"
adata.obs["cell_type_teaching"] = pd.Categorical(labels)

display(adata.obs["cell_type_teaching"].value_counts().to_frame("cells"))

For readable directed-pair results, we retain eight abundant compartments. Mast cells, NK
cells, adipocytes and ambiguous cells remain biologically interesting, but are excluded from
this short exercise. This is an analysis decision, not quality control.

In [ ]:
CELLTYPE_KEY = "cell_type_teaching"
KEEP_TYPES = [
    "Tumour luminal", "Tumour basal", "T cell", "B / plasma",
    "Myeloid", "Fibroblast", "Endothelial", "Perivascular",
]
adata = adata[adata.obs[CELLTYPE_KEY].isin(KEEP_TYPES)].copy()
adata.obs[CELLTYPE_KEY] = adata.obs[CELLTYPE_KEY].cat.remove_unused_categories()
xy = adata.obsm[SPATIAL_KEY]
palette = dict(zip(KEEP_TYPES, sns.color_palette("colorblind", len(KEEP_TYPES))))

fig, ax = plt.subplots(figsize=(9.5, 6.7))
for label in KEEP_TYPES:
    mask = adata.obs[CELLTYPE_KEY].to_numpy() == label
    ax.scatter(xy[mask, 0], xy[mask, 1], s=4, linewidths=0, alpha=0.8,
               color=palette[label], label=f"{label} ({mask.sum():,})", rasterized=True)
ax.set(aspect="equal", title="Marker-derived teaching annotations",
       xlabel="spatial x (µm)", ylabel="spatial y (µm)")
ax.invert_yaxis()
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False, markerscale=2)
plt.tight_layout()
plt.show()
print(f"LIANA+ object: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

## 3. Audit ligand–receptor coverage

The human LIANA+ `consensus` resource contains thousands of curated interactions. A targeted
313-gene Xenium panel can test a pair only when it measures the ligand, receptor and every
required complex subunit. Therefore, **an absent result usually means unmeasured or filtered,
not biologically absent**.

In [ ]:
RESOURCE_NAME = "consensus"
resource = li.rs.select_resource(RESOURCE_NAME)
measured_genes = set(adata.var_names)

def complex_is_measured(name):
    return set(str(name).split("_")) <= measured_genes

covered = resource[
    resource["ligand"].map(complex_is_measured) &
    resource["receptor"].map(complex_is_measured)
].drop_duplicates().copy()

print(f"resource interactions: {len(resource):,}")
print(f"fully measurable in this 313-gene panel: {len(covered):,}")
display(covered[["ligand", "receptor"]].drop_duplicates().head(15))

Only **23 resource rows** are fully measurable, so this is a focused demonstration rather
than a comprehensive breast-tumour interactome. Always report the tested coverage.

## 4. Define the spatial scale

We inspect the sixth-nearest-neighbour distance, approximately one local ring of cells. A
40 µm Gaussian bandwidth spans nearby cells while tolerating gaps between segmented cells.
The appropriate scale depends on platform geometry and signalling mechanism.

In [ ]:
distances = NearestNeighbors(n_neighbors=7).fit(xy).kneighbors(xy, return_distance=True)[0]
sixth = distances[:, 6]
BANDWIDTH = 40.0

for q in [10, 25, 50, 75, 90, 95]:
    print(f"{q:>2}th percentile: {np.percentile(sixth, q):.1f} µm")

fig, ax = plt.subplots(figsize=(6.8, 4.1))
ax.hist(sixth, bins=60, color="#4472C4", alpha=0.85)
ax.axvline(BANDWIDTH, color="#C1272D", linestyle="--", linewidth=2,
           label=f"bandwidth = {BANDWIDTH:.0f} µm")
ax.set(xlabel="distance to sixth-nearest cell (µm)", ylabel="cells",
       title="Choose the spatial scale from cell spacing")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
t0 = time.time()
pair_proximity = li.ut.spatial_pair_proximity(
    adata, groupby=CELLTYPE_KEY, kernel="gaussian", bandwidth=BANDWIDTH,
    spatial_key=SPATIAL_KEY, verbose=False,
)
print(f"Computed {len(pair_proximity)} directed proximities in {time.time()-t0:.1f} s")

prox_matrix = pair_proximity.pivot(index="source", columns="target", values="proximity")
prox_matrix = prox_matrix.reindex(index=KEEP_TYPES, columns=KEEP_TYPES)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(prox_matrix, cmap="mako", vmin=0, vmax=1, annot=True, fmt=".2f",
            linewidths=0.4, cbar_kws={"label": "LIANA+ proximity weight"}, ax=ax)
ax.set(title=f"Directed cell-type proximity ({BANDWIDTH:.0f} µm bandwidth)",
       xlabel="receiver / target", ylabel="sender / source")
plt.tight_layout()
plt.show()

## 5. Run LIANA+ consensus inference

We run the same consensus twice: expression-only, then spatially weighted. Smaller
`magnitude_rank` and `specificity_rank` values indicate stronger consensus evidence.
`expr_prop=0.05` requires every interaction component in at least 5% of the relevant cell
population. One hundred permutations are suitable for teaching; use more for stable research
inference.

In [ ]:
COMMON = dict(
    groupby=CELLTYPE_KEY, resource_name=RESOURCE_NAME,
    expr_prop=0.05, min_cells=20, use_raw=False,
    n_perms=100, seed=SEED, n_jobs=1, inplace=False, verbose=False,
)

t0 = time.time()
results_expression = li.mt.rank_aggregate(adata, **COMMON)
print(f"expression-only: {len(results_expression):,} rows in {time.time()-t0:.1f} s")

t0 = time.time()
results_spatial = li.mt.rank_aggregate(
    adata, spatial_key=SPATIAL_KEY,
    spatial_kwargs={"kernel": "gaussian", "bandwidth": BANDWIDTH,
                    "trim_fraction": 0.1},
    **COMMON,
)
print(f"spatially weighted: {len(results_spatial):,} rows in {time.time()-t0:.1f} s")
print("columns:", results_spatial.columns.tolist())

In [ ]:
cross = results_spatial.loc[results_spatial["source"] != results_spatial["target"]].copy()
cross["interaction"] = cross["ligand_complex"] + " → " + cross["receptor_complex"]
cross["cell_pair"] = cross["source"] + " → " + cross["target"]
top_cross = (
    cross.sort_values(["magnitude_rank", "specificity_rank"])
         .head(20)[["source", "target", "ligand_complex", "receptor_complex",
                    "magnitude_rank", "specificity_rank", "cellphone_pvals"]]
)
display(top_cross)

In [ ]:
plot_df = cross.sort_values(["magnitude_rank", "specificity_rank"]).head(18).copy()
eps = 1e-12
plot_df["magnitude_evidence"] = -np.log10(plot_df["magnitude_rank"].clip(lower=eps))
plot_df["specificity_evidence"] = -np.log10(plot_df["specificity_rank"].clip(lower=eps))
plot_df["label"] = plot_df["interaction"] + "\n" + plot_df["cell_pair"]
plot_df = plot_df.sort_values("magnitude_evidence")

fig, ax = plt.subplots(figsize=(10, 8))
points = ax.scatter(plot_df["magnitude_evidence"], plot_df["label"],
                    s=35 + 24 * plot_df["specificity_evidence"].clip(upper=8),
                    c=plot_df["magnitude_evidence"], cmap="viridis",
                    edgecolor="white", linewidth=0.5)
ax.set(xlabel="stronger consensus evidence (−log10 magnitude rank)", ylabel="",
       title="Spatially weighted cross-population candidates")
fig.colorbar(points, ax=ax, label="−log10 magnitude rank")
plt.tight_layout()
plt.show()

### 5.1 What did spatial weighting change?

We match identical cell-pair/interaction rows across runs. Negative log-rank changes mean a
candidate moved to a better rank after incorporating spatial proximity.

In [ ]:
ID_COLS = ["source", "target", "ligand_complex", "receptor_complex"]
comparison = results_expression[ID_COLS + ["magnitude_rank"]].merge(
    results_spatial[ID_COLS + ["magnitude_rank"]], on=ID_COLS,
    suffixes=("_expression", "_spatial"),
)
comparison["log10_rank_change"] = (
    np.log10(comparison["magnitude_rank_spatial"].clip(lower=1e-12)) -
    np.log10(comparison["magnitude_rank_expression"].clip(lower=1e-12))
)
display(comparison.loc[comparison["source"] != comparison["target"]]
                  .sort_values("log10_rank_change").head(15))

## 6. Where does an interaction occur?

Cell-type consensus gives a directed population-level result. LIANA+ local bivariate
analysis instead calculates a spatially weighted ligand–receptor score at every cell. We use
the same 40 µm scale so the two analyses make compatible neighbourhood assumptions.

In [ ]:
li.ut.spatial_neighbors(
    adata, bandwidth=BANDWIDTH, cutoff=0.1, max_neighbours=30,
    kernel="gaussian", set_diag=False, spatial_key=SPATIAL_KEY,
)
A = adata.obsp["spatial_connectivities"]
print(f"spatial graph: {A.nnz:,} directed edges; {A.nnz/adata.n_obs:.1f} per cell")

t0 = time.time()
lrdata = li.mt.bivariate(
    adata, resource_name=RESOURCE_NAME,
    local_name="cosine", global_name="morans", n_perms=100,
    mask_negatives=False, add_categories=True, nz_prop=0.05,
    use_raw=False, verbose=False,
)
print(f"local analysis: {lrdata.n_obs:,} cells × {lrdata.n_vars:,} interactions "
      f"in {time.time()-t0:.1f} s")
local_summary = lrdata.var.sort_values("morans", ascending=False)
display(local_summary.head(12))

`morans` summarises the spatial structure of each local interaction map. Positive values
indicate spatial co-clustering; values near zero indicate weak structure. It does not prove
that the receptor was activated.

In [ ]:
def dense_vector(matrix):
    return matrix.toarray().ravel() if sparse.issparse(matrix) else np.asarray(matrix).ravel()

def complex_expression(adata_obj, complex_name):
    parts = str(complex_name).split("_")
    return np.min(np.column_stack([dense_vector(adata_obj[:, g].X) for g in parts]), axis=1)

def plot_local_interaction(interaction):
    ligand = str(lrdata.var.loc[interaction, "ligand"])
    receptor = str(lrdata.var.loc[interaction, "receptor"])
    panels = [
        (complex_expression(adata, ligand), f"ligand: {ligand}", "viridis"),
        (complex_expression(adata, receptor), f"receptor: {receptor}", "viridis"),
        (dense_vector(lrdata[:, interaction].X), f"local weighted cosine: {interaction}", "magma"),
        (-np.log10(np.clip(dense_vector(lrdata[:, interaction].layers["pvals"]), 1e-4, 1)),
         "local permutation evidence (−log10 p)", "magma"),
    ]
    fig, axes = plt.subplots(2, 2, figsize=(11.5, 9))
    for ax, (values, title, cmap) in zip(axes.ravel(), panels):
        pts = ax.scatter(xy[:, 0], xy[:, 1], c=values, s=5, cmap=cmap,
                         linewidths=0, rasterized=True)
        ax.set_aspect("equal"); ax.invert_yaxis(); ax.set_title(title)
        ax.set_xticks([]); ax.set_yticks([])
        fig.colorbar(pts, ax=ax, fraction=0.045, pad=0.03)
    fig.suptitle("Expression and local spatial coordination", y=1.01, fontsize=14)
    plt.tight_layout()
    plt.show()

TOP_LOCAL = local_summary.index[0]
print("top local interaction:", TOP_LOCAL)
plot_local_interaction(TOP_LOCAL)

### 6.1 Named breast-tumour microenvironment example: CXCL12–CXCR4

CXCL12–CXCR4 is measurable in this panel and resource. The local map shows where expression
and neighbourhood geometry align. It does **not** establish signalling direction at each
cell, downstream pathway activity or whether the effect is tumour-promoting.

In [ ]:
NAMED_INTERACTION = "CXCL12^CXCR4"
if NAMED_INTERACTION in lrdata.var_names:
    display(lrdata.var.loc[[NAMED_INTERACTION]])
    plot_local_interaction(NAMED_INTERACTION)
else:
    print(NAMED_INTERACTION, "was filtered under the current settings.")

## 7. Sensitivity to neighbourhood scale

The bandwidth is a biological assumption. Compare nearby scales; do not select the one that
produces the most exciting result. A contact/juxtacrine mechanism generally warrants a
shorter scale than a diffusible ligand.

In [ ]:
bandwidths = [25.0, 40.0, 60.0]
focus_pairs = [
    ("Fibroblast", "Tumour luminal"),
    ("Fibroblast", "T cell"),
    ("Endothelial", "Perivascular"),
]
rows = []
for bw in bandwidths:
    prox = li.ut.spatial_pair_proximity(
        adata, groupby=CELLTYPE_KEY, kernel="gaussian", bandwidth=bw,
        spatial_key=SPATIAL_KEY, verbose=False,
    )
    for source, target in focus_pairs:
        value = prox.loc[(prox["source"] == source) & (prox["target"] == target),
                         "proximity"].iloc[0]
        rows.append({"bandwidth_um": bw, "source": source, "target": target,
                     "proximity": value})
sensitivity = pd.DataFrame(rows)
sensitivity["cell_pair"] = sensitivity["source"] + " → " + sensitivity["target"]
display(sensitivity)

fig, ax = plt.subplots(figsize=(7.8, 4.4))
sns.lineplot(data=sensitivity, x="bandwidth_um", y="proximity", hue="cell_pair",
             marker="o", linewidth=2.2, ax=ax)
ax.set(xlabel="Gaussian bandwidth (µm)", ylabel="LIANA+ proximity",
       title="Neighbourhood-scale sensitivity")
ax.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 8. Save the results

The archive contains the expression-only and spatial consensus tables, cell-type proximity,
local-interaction summary, coverage table and marker-derived annotations.

In [ ]:
import shutil

OUTPUT_DIR = Path("liana_breast_xenium_results")
OUTPUT_DIR.mkdir(exist_ok=True)
results_spatial.to_csv(OUTPUT_DIR / "liana_spatial_rank_aggregate.csv", index=False)
results_expression.to_csv(OUTPUT_DIR / "liana_expression_only_rank_aggregate.csv", index=False)
pair_proximity.to_csv(OUTPUT_DIR / "celltype_spatial_proximity.csv", index=False)
lrdata.var.to_csv(OUTPUT_DIR / "local_bivariate_interaction_summary.csv")
covered.to_csv(OUTPUT_DIR / "measurable_consensus_resource_rows.csv", index=False)
adata.obs[[CELLTYPE_KEY]].to_csv(OUTPUT_DIR / "teaching_annotations.csv")
archive = shutil.make_archive("liana_breast_xenium_results", "zip", OUTPUT_DIR)
print("saved:", archive)

if IN_COLAB:
    from google.colab import files
    files.download(archive)

## 9. Interpretation checklist

Before reporting an interaction, ask:

1. **Coverage:** were the ligand, receptor and all complex subunits measured?
2. **Identity:** are sender and receiver annotations independently credible?
3. **Expression:** are genes detected in enough cells, rather than a few outliers?
4. **Proximity:** is the neighbourhood scale appropriate for the signalling mechanism?
5. **Localisation:** does the local signal occupy a coherent tissue region?
6. **Direction and biology:** does prior evidence support the proposed sender–receiver order?
7. **Validation:** can protein imaging, perturbation or downstream pathway evidence test it?

### Transfer to your own Xenium/Visium data

Your `AnnData` needs normalised/log-transformed expression, a trusted cell-type column in
`.obs`, coordinates in `.obsm['spatial']`, and gene symbols matching the selected LIANA+
resource. For Visium spots, remember that each spot mixes cells: use a defensible spot label,
cell-type abundance model or deconvolution, and phrase results as spot/neighbourhood-level
associations rather than direct single-cell conversations.

### Sources

- [LIANA+ framework and spatial analysis paper](https://www.nature.com/articles/s41556-024-01469-w)
- [Janesick *et al.*, breast-cancer Xenium study](https://www.nature.com/articles/s41467-023-43458-x)
- [Compact replicate-2 `.h5ad` download page](https://huggingface.co/datasets/Shaow/humanbreast_xenium_janesick)
- [LIANA+ documentation](https://liana-py.readthedocs.io/)